In [2]:
pip install pyspark

In [3]:
import pyspark
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.window import Window

In [5]:
spark = SparkSession.builder.getOrCreate()

In [7]:
df_video = spark.read.parquet('videos-preparados.snappy.parquet', header=True, inferSchema=True)

df_video.show(5)
df_video.printSchema()

+--------------------+-----------+------------+-------+------+--------+--------+-----------+----+-----+-------------+--------------------+--------------------+--------------------+
|               Title|   Video ID|Published At|Keyword| Likes|Comments|   Views|Interaction|Year|Month|Keyword Index|        Features PCA|     Features Normal|            Features|
+--------------------+-----------+------------+-------+------+--------+--------+-----------+----+-----+-------------+--------------------+--------------------+--------------------+
|ASMR MUKBANG DOUB...|--ZI0dSbbNU|  2020-04-18|mukbang|378858|   18860|17975269|   18372987|2020|    4|         30.0|[0.6985786560867407]|[0.02303716158264...|[378858.0,1.79752...|
|Deadly car bomb d...|--hxd1CrOqg|  2022-08-22|   news|  6379|    4853|  808787|     820019|2022|    8|         37.0|[0.8936407990235931]|[3.87946679100418...|[6379.0,808787.0,...|
|How Biden&#39;s s...|--ixiTypG8g|  2022-08-24|   news|  1029|    2347|   97434|     100810|202

In [8]:
contagem_keyword = df_video.groupBy('Keyword').agg(count('*').alias('Keyword_contagem'))
contagem_keyword.show()

+----------------+----------------+
|         Keyword|Keyword_contagem|
+----------------+----------------+
|computer science|              48|
|            lofi|              40|
|         finance|              39|
|             cnn|              50|
|           apple|              42|
|            news|              39|
|         mukbang|              45|
|       education|              24|
|       interview|              50|
|          crypto|              50|
|   mathchemistry|              15|
|            food|              48|
|    data science|              50|
|        trolling|              50|
|        tutorial|              50|
|      literature|              46|
|             sat|              49|
|         history|              49|
|           cubes|              49|
|           music|              46|
+----------------+----------------+
only showing top 20 rows



In [9]:
df_video.agg(countDistinct(col('Keyword')).alias('Distinct_Keywords')).show()

+-----------------+
|Distinct_Keywords|
+-----------------+
|               41|
+-----------------+



In [10]:
media_interaction = df_video.groupBy('Keyword').agg(avg('Interaction').alias('Media_Interaction'))
media_interaction.show()

+----------------+--------------------+
|         Keyword|   Media_Interaction|
+----------------+--------------------+
|computer science|  1226793.0208333333|
|            lofi|         4167085.875|
|         finance|   708542.9487179487|
|             cnn|           570650.86|
|           apple|1.0873628214285715E7|
|            news|  251688.71794871794|
|         mukbang|1.1053630377777778E7|
|       education|         2750838.625|
|       interview|          3044867.04|
|          crypto|            413676.2|
|   mathchemistry|  3427342.7333333334|
|            food|   5352944.104166667|
|    data science|           562465.28|
|        trolling|          1484584.88|
|        tutorial|           6936688.3|
|      literature|            881726.5|
|             sat|           1098927.0|
|         history| 1.565269257142857E7|
|           cubes|1.5043961224489795E7|
|           music|2.9691370304347824E7|
+----------------+--------------------+
only showing top 20 rows



In [11]:
rank_interaction = df_video.groupBy('Keyword').agg(
    max('Interaction').alias('Rank Interactions')
).orderBy(col('Rank Interactions').desc())

rank_interaction.show()

+--------+-----------------+
| Keyword|Rank Interactions|
+--------+-----------------+
| animals|       1593623628|
|   music|        922551152|
|     bed|        532691631|
| history|        440187490|
|   apple|        429916936|
| mrbeast|        300397699|
|  google|        239385460|
|business|        210025196|
|   cubes|        170925917|
|  sports|        106924567|
| mukbang|         87433858|
|    lofi|         86445177|
|tutorial|         69616442|
|  movies|         65253870|
|  marvel|         56247330|
|  how-to|         53053975|
|    food|         48754479|
| physics|         43463298|
|    asmr|         34411125|
|nintendo|         32268486|
+--------+-----------------+
only showing top 20 rows



In [12]:
media_var_views = df_video.groupBy('Keyword').agg(
    avg('Views').alias('Media Views'),
    var_samp('Views').alias('Variância Views')
)

media_var_views.show()

+----------------+--------------------+--------------------+
|         Keyword|         Media Views|     Variância Views|
+----------------+--------------------+--------------------+
|computer science|  1191958.7083333333| 2.81219868165102E12|
|            lofi|           4089363.0|1.846209641476677...|
|         finance|   694223.4358974359|3.304483175097042...|
|             cnn|           554240.38|1.563423618468118...|
|           apple|1.0746930452380951E7|4.299927977442589E15|
|            news|   247492.1794871795|1.067512576672564...|
|         mukbang|1.0904772355555555E7|5.586073238973179...|
|       education|  2684432.8333333335|1.833572249339214...|
|       interview|          2966111.86|1.819220996034335E13|
|          crypto|           404608.22|3.513691634369074E12|
|   mathchemistry|  3328125.2666666666|2.491467065256849...|
|            food|          5252406.25|7.326374128154842E13|
|    data science|           544771.98|5.479336525349994...|
|        trolling|      

In [15]:
max_min_med_views = df_video.groupBy('Keyword').agg(
    format_number(max('Views'),0).alias('Max Views'),
    format_number(min('Views'),0).alias('Min Views'),
    format_number(avg('Views'),0).alias('Media Views')
)

max_min_med_views.show()

+----------------+-----------+---------+-----------+
|         Keyword|  Max Views|Min Views|Media Views|
+----------------+-----------+---------+-----------+
|computer science|  7,004,107|   16,115|  1,191,959|
|            lofi| 84,747,957|    6,817|  4,089,363|
|         finance|  9,450,554|    1,195|    694,223|
|             cnn|  1,889,320|   51,269|    554,240|
|           apple|425,478,119|    1,954| 10,746,930|
|            news|  1,465,011|   10,642|    247,492|
|         mukbang| 86,169,225|    3,618| 10,904,772|
|       education| 17,103,736|    6,611|  2,684,433|
|       interview| 22,529,756|    2,587|  2,966,112|
|          crypto| 11,805,668|    1,599|    404,608|
|   mathchemistry| 18,496,859|       25|  3,328,125|
|            food| 48,018,833|   47,430|  5,252,406|
|    data science|  3,069,097|      911|    544,772|
|        trolling| 14,286,302|    5,388|  1,420,141|
|        tutorial| 68,512,549|   19,323|  6,761,032|
|      literature|  4,231,789|    2,847|    86

In [18]:
df_ordenado = df_video.orderBy('Published At')
df_ordenado.show(5)

+--------------------+-----------+------------+---------+-------+--------+---------+-----------+----+-----+-------------+--------------------+--------------------+--------------------+
|               Title|   Video ID|Published At|  Keyword|  Likes|Comments|    Views|Interaction|Year|Month|Keyword Index|        Features PCA|     Features Normal|            Features|
+--------------------+-----------+------------+---------+-------+--------+---------+-----------+----+-----+-------------+--------------------+--------------------+--------------------+
|J. Holiday - Bed ...|82t_UOMHPJY|  2007-07-16|      bed| 515049|   16568| 78137822|   78669439|2007|    7|         31.0|[0.8176384282479404]|[0.03131848543427...|[515049.0,7.81378...|
|     J Holiday - Bed|uAwPjFq1W4s|  2007-12-11|      bed| 119272|    3743| 26826274|   26949289|2007|   12|         31.0|[0.8576354497275425]|[0.00725259627842...|[119272.0,2.68262...|
|Pink Floyd - We D...|MAe_w9a_IN8|  2008-07-25|education|  50074|    3095| 

In [19]:
data_published = df_ordenado.groupBy('Keyword').agg(
    first('Published At').alias('Primeiro Published At'),
    last('Published At').alias('Ultimo Published At')
)
data_published.show()

+----------------+---------------------+-------------------+
|         Keyword|Primeiro Published At|Ultimo Published At|
+----------------+---------------------+-------------------+
|computer science|           2009-08-20|         2022-08-12|
|            lofi|           2019-12-08|         2022-08-24|
|         finance|           2012-11-27|         2022-08-24|
|             cnn|           2022-07-14|         2022-08-24|
|           apple|           2016-11-02|         2022-08-24|
|            news|           2022-08-18|         2022-08-24|
|       education|           2008-07-25|         2022-08-24|
|         mukbang|           2020-02-29|         2022-08-24|
|       interview|           2016-01-05|         2022-08-24|
|          crypto|           2022-03-11|         2022-08-24|
|   mathchemistry|           2013-04-15|         2022-05-03|
|            food|           2017-05-31|         2022-08-24|
|    data science|           2018-06-23|         2022-08-24|
|        trolling|      

In [25]:
contagem_title = df_video.select(count(col('Title')).alias('Total Titles')).show()

+------------+
|Total Titles|
+------------+
|        1869|
+------------+



In [28]:
contagem_title_distintos = df_video.select(count_distinct('Title').alias('Titulos Unicos'))

contagem_title_distintos.show()

+--------------+
|Titulos Unicos|
+--------------+
|          1854|
+--------------+



In [36]:
df_video.withColumn('Ano', year('Published At')).groupBy('Ano').agg(count('Title').alias('Qtd de Registros')).orderBy('Ano').show()

+----+----------------+
| Ano|Qtd de Registros|
+----+----------------+
|2007|               2|
|2008|               1|
|2009|               9|
|2010|               6|
|2011|               4|
|2012|              12|
|2013|               6|
|2014|              10|
|2015|              15|
|2016|              34|
|2017|              47|
|2018|              57|
|2019|              86|
|2020|             158|
|2021|             229|
|2022|            1193|
+----+----------------+



In [39]:
df_video.withColumn('Ano', year('Published At')).withColumn('Mes', month('Published At')).groupBy('Ano', 'Mes').agg(count('Title').alias('Qtd de Registros')).orderBy('Ano', 'Mes').show(50)

+----+---+----------------+
| Ano|Mes|Qtd de Registros|
+----+---+----------------+
|2007|  7|               1|
|2007| 12|               1|
|2008|  7|               1|
|2009|  2|               2|
|2009|  6|               2|
|2009|  7|               1|
|2009|  8|               1|
|2009| 10|               1|
|2009| 12|               2|
|2010|  3|               1|
|2010|  5|               2|
|2010|  6|               1|
|2010|  9|               1|
|2010| 10|               1|
|2011|  2|               1|
|2011|  5|               1|
|2011|  9|               1|
|2011| 10|               1|
|2012|  1|               1|
|2012|  2|               3|
|2012|  3|               2|
|2012|  4|               1|
|2012|  6|               1|
|2012|  8|               1|
|2012| 10|               1|
|2012| 11|               2|
|2013|  3|               1|
|2013|  4|               2|
|2013|  5|               2|
|2013|  6|               1|
|2014|  2|               1|
|2014|  4|               2|
|2014|  7|          

In [42]:
media_cumulativa = Window.partitionBy('Keyword').orderBy('Likes').rowsBetween(Window.unboundedPreceding, Window.currentRow)

df_video = df_video.withColumn('Media Cumulativa de Likes', avg('Likes').over(media_cumulativa))

df_video.show()

+--------------------+-----------+------------+-------+-----+--------+-------+-----------+----+-----+-------------+--------------------+--------------------+--------------------+------------------+-------------------------+
|               Title|   Video ID|Published At|Keyword|Likes|Comments|  Views|Interaction|Year|Month|Keyword Index|        Features PCA|     Features Normal|            Features|  Media_Cumulativa|Media Cumulativa de Likes|
+--------------------+-----------+------------+-------+-----+--------+-------+-----------+----+-----+-------------+--------------------+--------------------+--------------------+------------------+-------------------------+
|SAY HALO To Anima...|9QW7CORIoFw|  2022-07-02|animals|   37|       1|  31343|      31381|2022|    7|         38.0| [0.910444545590002]|[2.31065420153854...|[37.0,31343.0,202...|              37.0|                     37.0|
|AWW SO CUTE! Cute...|1mQqbvu0gqo|  2022-08-24|animals|  371|      19|  23448|      23838|2022|    8|   

In [44]:
spark.stop()